# How to use the Matcher framework?

The `Matcher` framework provides a high-level interface for computing correspondences between shapes. It encapsulates the full functional map pipeline into a single, configurable class.

In [1]:
import gsops.backend as gs

from geomfum.dataset import NotebooksDataset
from geomfum.matcher import (
    FeatureMatcher,
    FunctionalMapMatcher,
    QuickFunctionalMapMatcher,
)
from geomfum.shape import TriangleMesh

[Load meshes](00_load_mesh_from_file.ipynb).

In [2]:
dataset = NotebooksDataset()

mesh_a = TriangleMesh.from_file(dataset.get_filename("faust-00"))
mesh_b = TriangleMesh.from_file(dataset.get_filename("faust-04"))

mesh_a.n_vertices, mesh_b.n_vertices

INFO:root:Data has already been downloaded... using cached file ('C:\Users\giuli\.geomfum\data\faust-00.off').
INFO:root:Data has already been downloaded... using cached file ('C:\Users\giuli\.geomfum\data\faust-04.off').


(6890, 6890)

## Feature Matcher

The simplest way to match shapes is computing features and performing nearest neighbor search, this routine is made by the Feature matcher.



In [3]:
# Basic usage with defaults
matcher = FeatureMatcher()
result = matcher(mesh_a, mesh_b)
p2p21 = result.p2p21  # Maps each vertex in B to a vertex in A

print(f"P2P21 shape: {p2p21.shape}")

P2P21 shape: (6890,)


The result contains:
- `p2p21`: point-to-point correspondence from B to A (for each vertex in B, gives corresponding vertex in A)
- `descr_a`, `descr_b`: computed descriptors
- `fmap12`: functional map matrix from A to B (None for FeatureMatcher)
- `refined_fmap12`: refined functional map (None for FeatureMatcher)

In [4]:
result.descr_a.shape, result.descr_b.shape

((200, 6890), (200, 6890))

## Functional Map Matcher

For more robust matching, use `FunctionalMapMatcher` which optimizes a functional map.

In [5]:
matcher = FunctionalMapMatcher()
result = matcher(mesh_a, mesh_b)

print(f"P2P21 shape: {result.p2p21.shape}")  # B -> A
print(f"Fmap12 shape: {result.fmap12.shape}")  # A -> B
print(f"Refined Fmap12 shape: {result.refined_fmap12.shape}")

P2P21 shape: (6890,)
Fmap12 shape: (30, 30)
Refined Fmap12 shape: (30, 30)


## Using landmarks

[Set landmarks](./06_landmarks.ipynb) on both shapes for better matching.

In [6]:
mesh_a.set_landmarks(gs.array([412, 5891, 6593, 3323, 2119]))
mesh_b.set_landmarks(gs.array([412, 5891, 6593, 3323, 2119]))

Use landmarks by adding `LandmarkWaveKernelSignature` to the descriptors list.

In [7]:
from geomfum.descriptor.pipeline import ArangeSubsampler, DescriptorPipeline
from geomfum.descriptor.spectral import LandmarkWaveKernelSignature, WaveKernelSignature

matcher = FunctionalMapMatcher(
    descriptor_pipeline=DescriptorPipeline(
        [
            WaveKernelSignature.from_registry(n_domain=200),
            LandmarkWaveKernelSignature.from_registry(n_domain=200),
            ArangeSubsampler(subsample_step=10),
        ]
    ),
)
result = matcher(mesh_a, mesh_b)

result.p2p21.shape

(6890,)

## Preset matchers

Several preset matchers are available for common use cases.

### QuickMatcher

Fast matching with reduced settings (smaller spectrum, fewer refinement iterations).

In [8]:
quick_matcher = QuickFunctionalMapMatcher()

result = quick_matcher(mesh_a, mesh_b)

result.p2p21.shape

(6890,)

## Custom configuration

Use `MatcherConfig` to fully customize the matching pipeline.

In [9]:
from geomfum.matcher import FunctionalMapMatcher
from geomfum.refine import IcpRefiner, RefinementPipeline, ZoomOut

fmap_size = 20  # Size of functional map matrix
descriptor_pipeline = DescriptorPipeline(
    [
        WaveKernelSignature(n_domain=100),
        LandmarkWaveKernelSignature(n_domain=200),
        ArangeSubsampler(subsample_step=10),
    ]
)
refiners = RefinementPipeline(
    [  # Custom refinement pipeline
        IcpRefiner(nit=5),
        ZoomOut(nit=4, step=5),
    ],
)

custom_matcher = FunctionalMapMatcher(
    fmap_size=fmap_size,
    descriptor_pipeline=descriptor_pipeline,
    refiner=refiners,
)
result = custom_matcher(mesh_a, mesh_b)

result.p2p21.shape

(6890,)

## Further reading

* [How to compute a functional map?](./07_functional_map.ipynb)

* [How to refine a functional map?](./15_refine_functional_map.ipynb)

* [How to create a descriptor pipeline?](./04_descriptor_pipeline.ipynb)

* [How to set landmarks?](./06_landmarks.ipynb)